# AI Invoice — Post-Merge GGUF Conversion

Use this notebook after a **Colab runtime restart**.
- Merged model is read from **Google Drive**
- Final `.gguf` is written back to **Google Drive** — no `/content` disk pressure

Run all cells **top-to-bottom**.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Set Paths & Verify Model Files

Update `MODEL_PATH` here if you move the folder on Drive.

In [ ]:
import os

MODEL_PATH   = "/content/drive/MyDrive/invoice-models/invoice-qwen2.5-1.5b-merged"
DRIVE_OUTPUT = "/content/drive/MyDrive/invoice-models/invoice-qwen2.5-1.5b-q8_0.gguf"

# Make env vars available to shell cells (Steps 5-6)
os.environ["MODEL_PATH"]   = MODEL_PATH
os.environ["DRIVE_OUTPUT"] = DRIVE_OUTPUT

print("Checking model files in Drive...")
for f in sorted(os.listdir(MODEL_PATH)):
    size_mb = os.path.getsize(os.path.join(MODEL_PATH, f)) / 1e6
    print(f"  {f:<50} {size_mb:>8.1f} MB")

## Step 3 — Install Dependencies

In [ ]:
# Only transformers is needed here.
# llama.cpp requirements are handled in Step 5.
!pip install -q transformers

## Step 4 — Fix Tokenizer

> **Run 4a, then 4b — order matters.**
> 4a downloads and overwrites the tokenizer files on Drive.
> 4b patches `tokenizer_config.json` that 4a just wrote.

In [ ]:
# ── 4a: Overwrite with clean Qwen tokenizer ──────────────────
from transformers import AutoTokenizer

print("Downloading clean tokenizer from Qwen/Qwen2.5-1.5B-Instruct...")
clean_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
clean_tok.save_pretrained(MODEL_PATH)
print("Clean tokenizer saved to Drive model folder")

In [ ]:
# ── 4b: Patch extra_special_tokens in tokenizer_config.json ──
import json, pathlib

cfg_path = pathlib.Path(MODEL_PATH) / "tokenizer_config.json"
cfg      = json.loads(cfg_path.read_text())

val = cfg.get("extra_special_tokens")
print(f"Before: type={type(val).__name__}, value={val}")

if isinstance(val, list):
    cfg["extra_special_tokens"] = {}
    cfg_path.write_text(json.dumps(cfg, indent=2, ensure_ascii=False))
    print("Patched: extra_special_tokens is now {}")
else:
    print("No patch needed — already a dict or absent")

## Step 5 — Clone llama.cpp & Install Requirements

`--depth 1` keeps the clone small (~50 MB) — important on free-tier Colab disk.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
print("llama.cpp ready")

## Step 6 — Convert to GGUF

Reads directly from Drive → writes directly to Drive.
Expected time: **5–10 min** for a 1.5B model (Drive I/O is the bottleneck).

In [ ]:
# {MODEL_PATH} and {DRIVE_OUTPUT} are IPython variable expansion —
# they resolve to the Python variables set in Step 2.
!python /content/llama.cpp/convert_hf_to_gguf.py {MODEL_PATH} --outfile {DRIVE_OUTPUT} --outtype q8_0

## Step 7 — Verify Output

In [ ]:
size_gb = os.path.getsize(DRIVE_OUTPUT) / 1e9
print(f"GGUF saved!")
print(f"   Path : {DRIVE_OUTPUT}")
print(f"   Size : {size_gb:.2f} GB")
print()
if size_gb < 1.4:
    print("WARNING: File seems too small — conversion may have been cut short.")
else:
    print("Size looks correct for Qwen2.5-1.5B at q8_0 (~1.6-1.7 GB expected).")